# Assignment 1A — Building & Fine-Tuning a Domain-Specific LLM

**Domain (Variant 1, default):** Medical & Clinical Literature — Type 2 Diabetes Management
**Model:** `microsoft/biogpt-large` (347M, ungated)
**Team:** P1 Corpus & Instruction Data · P2 Packing & CPT Training · P3 Model Audit & Evaluation · P4 QLoRA & Submission

This notebook follows `SPRINT-PLAN.md` step by step. Each section below matches one graded step.

## Prerequisites — Environment Setup

Installs anything missing and prints the exact versions used, so the graded output
records the environment that produced the results below.

Safe to re-run: packages already present are skipped, so a *Restart kernel → Run all*
pass costs nothing on a machine that is already set up.

In [8]:
# ---------------------------------------------------------------------------
# Install any missing packages.
#
# The install is split into two tiers on purpose. This notebook runs in two very
# different places: a plain CPU machine for Step 1, and a GPU runtime (free Colab
# T4 / BITS A100) for Steps 3-5 and Part B.
#
#   Tier 1 (always)  - Step 1 only needs two small, pure-Python packages.
#   Tier 2 (GPU box) - peft / trl / bitsandbytes all declare torch as a dependency,
#                      so installing them on a machine with no torch would drag in
#                      an ~800 MB CPU-only build that is useless for training and
#                      may later conflict with the runtime's CUDA-matched wheel.
#                      They are therefore installed only where torch already exists.
# ---------------------------------------------------------------------------
import importlib.util
import subprocess
import sys

# pip name -> import name. They coincide for everything here, but stating the
# mapping explicitly means a future package whose names differ (e.g.
# scikit-learn / sklearn) can be added without changing the logic.
STEP1_PACKAGES = {
    "pypdf": "pypdf",            # page-by-page PDF text extraction
    "langdetect": "langdetect",  # English-only language filter
}

TRAINING_PACKAGES = {
    "transformers": "transformers",  # Steps 2-4: tokenizer, model, Trainer
    "accelerate": "accelerate",      # required by transformers.Trainer
    "pyarrow": "pyarrow",            # Step 2: Parquet writer
    "pandas": "pandas",              # Step 2: dataframe around packed sequences
    "matplotlib": "matplotlib",      # Step 4: loss curve
    "sacremoses": "sacremoses",      # REQUIRED by biogpt-large's Moses tokenizer
    "peft": "peft",                  # Part B: LoRA adapters
    "bitsandbytes": "bitsandbytes",  # Part B: 4-bit (nf4) quantization
    "trl": "trl",                    # Part B: SFTTrainer
    "datasets": "datasets",          # Part B: loading instruction_dataset.jsonl
}


def is_installed(import_name: str) -> bool:
    """Presence check via find_spec: cheap, and does not execute the module."""
    return importlib.util.find_spec(import_name) is not None


def pip_install(packages: list[str]) -> None:
    """Install into the interpreter running THIS notebook.

    sys.executable -m pip, never a bare `pip`: on machines with several Python
    installs, a bare `pip` can install into a different interpreter than the one
    the kernel is using, leaving the import still failing after a "successful"
    install.
    """
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


# --- Tier 1: always required -------------------------------------------------
missing_step1 = [p for p, mod in STEP1_PACKAGES.items() if not is_installed(mod)]
if missing_step1:
    print(f"Installing Step 1 packages: {', '.join(missing_step1)}")
    pip_install(missing_step1)
    print("  done")
else:
    print("Step 1 packages: already present")

# --- Tier 2: only on a machine that already has torch ------------------------
missing_training = [p for p, mod in TRAINING_PACKAGES.items() if not is_installed(mod)]

if not is_installed("torch"):
    # No torch => this is the CPU box used for Step 1. Report and move on rather
    # than pulling the training stack (and a CPU-only torch) onto it.
    print("\ntorch not found - skipping the training stack (Steps 2+).")
    if missing_training:
        print(f"  pending on the GPU runtime: {', '.join(missing_training)}")
    print("  Install torch with the build matching your runtime's CUDA version;")
    print("  do not `pip install torch` over Colab's preinstalled wheel.")
elif missing_training:
    print(f"\nInstalling training packages: {', '.join(missing_training)}")
    pip_install(missing_training)
    print("  done")
else:
    print("\nTraining packages: already present")

Step 1 packages: already present

Training packages: already present


In [9]:
# ---------------------------------------------------------------------------
# Environment report.
#
# Printed into the notebook so the submitted output records exactly which
# Python, package versions and GPU produced the results below - which is what
# makes the run reproducible by whoever marks it.
# ---------------------------------------------------------------------------
import importlib.metadata as md
import platform

print(f"Python {platform.python_version()} | {platform.system()} {platform.machine()}")
print("-" * 56)

for pip_name in {**STEP1_PACKAGES, **TRAINING_PACKAGES}:
    try:
        print(f"  {pip_name:<16} {md.version(pip_name)}")
    except md.PackageNotFoundError:
        # Expected for the training stack while running Step 1 on a CPU machine.
        print(f"  {pip_name:<16} -")

# GPU check. Step 1 runs fine on CPU; Steps 3-4 and Part B do not.
print("-" * 56)
try:
    import torch

    print(f"  {'torch':<16} {torch.__version__}")
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  {'GPU':<16} {name} ({vram:.1f} GB)")
    else:
        print(f"  {'GPU':<16} none visible - OK for Step 1, required from Step 3 on")
except ImportError:
    print(f"  {'torch':<16} - (required from Step 3 on)")

Python 3.13.15 | Linux x86_64
--------------------------------------------------------
  pypdf            6.17.0
  langdetect       1.0.9
  transformers     5.16.1
  accelerate       1.14.0
  pyarrow          25.0.1
  pandas           2.2.3
  matplotlib       3.10.0
  sacremoses       0.2.0
  peft             0.20.0
  bitsandbytes     0.50.2
  trl              1.12.0
  datasets         5.0.1
--------------------------------------------------------
  torch            2.11.0+cu128
  GPU              Tesla T4 (14.6 GB)


## Step 1 — Data Collection, Extraction & Cleaning (2 marks · P1)

**Produces:** `domain_corpus/*.txt`, `cleaning_stats.json`

Pipeline: page-by-page extraction → boilerplate/header-footer stripping (our added step) →
length filter → deduplication → language filter.

In [10]:
# ---------------------------------------------------------------------------
# Step 1 setup: imports and tunable configuration.
#
# Every threshold below is a deliberate choice we have to justify in the Step 1
# report, so they all live here rather than being scattered as magic numbers
# further down.
# ---------------------------------------------------------------------------
from __future__ import annotations

import hashlib                    # exact-duplicate detection via content hashing
import json                       # writing cleaning_stats.json
import re                         # whitespace / hyphenation normalisation
from collections import Counter   # counting how many pages each line appears on
from pathlib import Path

from pypdf import PdfReader       # brief allows "any standard PDF extraction library"

# --- paths -----------------------------------------------------------------
RAW_PDF_DIR = Path("raw_pdfs")                    # input:  the source PDFs
OUT_DIR = Path("domain_corpus")                   # output: cleaned .txt (graded deliverable)
STATS_PATH = Path("outputs/cleaning_stats.json")  # output: per-stage counts

# Create every directory the notebook reads from or writes to, so a fresh clone of
# the repo runs without manual setup. mkdir(exist_ok=True) is idempotent, so this is
# safe on every re-run; parents=True means a missing "outputs/" is created too.
RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
STATS_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- cleaning thresholds ---------------------------------------------------
MIN_CHARS = 1_000             # length filter: below ~1k chars a "document" is a cover
                              # page or a failed extraction, not usable training text

NEAR_DUP_THRESHOLD = 0.85     # Jaccard overlap above which two documents are treated as
                              # duplicates. 0.85 catches revised editions of the same
                              # guideline without merging genuinely different documents
                              # published by the same body.

SHINGLE_SIZE = 5              # words per shingle. 5 is long enough that stock medical
                              # phrases ("type 2 diabetes mellitus") don't inflate the
                              # similarity score between unrelated documents.

LANGUAGE = "en"               # brief: "retain only English-language documents"

BOILERPLATE_PAGE_RATIO = 0.6  # a line appearing on >=60% of pages is page furniture
                              # (running header / footer), not content

SEED = 42                     # langdetect is non-deterministic unless seeded

In [11]:
class Pipeline:
    """Applies cleaning stages to a {document_name: text} mapping, recording the
    before AND after figures for every stage.

    The brief asks for "document counts before and after each cleaning step" and
    for "which step had the greatest impact on corpus size". Each stage therefore
    stores both sides of the transition explicitly rather than leaving the "before"
    implicit in the previous row - so the recorded numbers answer the question in
    the same shape it was asked.

    Two kinds of stage exist, and they are measured differently:
      * apply()     - a filter that DROPS whole documents -> impact in documents
      * transform() - rewrites text, dropping no documents -> impact in characters,
                      since its document count never changes and would always
                      report an impact of zero
    """

    def __init__(self, docs: dict[str, str]):
        self.docs = docs
        # The baseline row. It has no "before" because nothing preceded it.
        self.stages: list[dict] = [{
            "stage": "raw extraction (page-by-page)",
            "kind": "input",
            "documents_before": None,
            "documents_after": len(docs),
            "documents_removed": 0,
            "characters_before": None,
            "characters_after": self._chars(),
            "characters_removed": 0,
        }]

    def _chars(self) -> int:
        return sum(len(t) for t in self.docs.values())

    def _record(self, name: str, kind: str, docs_before: int, chars_before: int) -> None:
        self.stages.append({
            "stage": name,
            "kind": kind,
            "documents_before": docs_before,
            "documents_after": len(self.docs),
            "documents_removed": docs_before - len(self.docs),
            "characters_before": chars_before,
            "characters_after": self._chars(),
            "characters_removed": chars_before - self._chars(),
        })

    def apply(self, name: str, keep_fn) -> None:
        """Drop every document for which keep_fn(name, text) returns False."""
        docs_before, chars_before = len(self.docs), self._chars()
        self.docs = {k: v for k, v in self.docs.items() if keep_fn(k, v)}
        self._record(name, "filter", docs_before, chars_before)

    def transform(self, name: str, map_fn) -> None:
        """Rewrite every document's text via map_fn(name, text), keeping all documents."""
        docs_before, chars_before = len(self.docs), self._chars()
        self.docs = {k: map_fn(k, v) for k, v in self.docs.items()}
        self._record(name, "transform", docs_before, chars_before)

    def biggest_impact(self) -> dict:
        """Return the stage with the greatest impact, for the brief's report question.

        Document-count impact wins when any filter actually dropped something, since
        that is what "impact on corpus size" most directly means. Otherwise it falls
        back to characters, so a corpus where nothing needs filtering still names the
        stage that did the real cleaning work.
        """
        graded = self.stages[1:]
        by_docs = max(graded, key=lambda s: s["documents_removed"], default=self.stages[0])
        if by_docs["documents_removed"] > 0:
            return by_docs
        return max(graded, key=lambda s: s["characters_removed"], default=self.stages[0])

In [12]:
# ---------------------------------------------------------------------------
# Extraction and text-level cleaning.
#
# The three cleaning operations below are kept as SEPARATE functions so the
# pipeline can measure each one independently. Bundling them would let one stage
# take credit for another's work, which matters because the brief grades "which
# step had the greatest impact".
# ---------------------------------------------------------------------------

def extract_pages(pdf_path: Path) -> list[str]:
    """Extract text page-by-page, as the brief explicitly requires.

    Returns one string per page (empty string for pages that yield no text).
    Keeping pages separate instead of concatenating immediately is what makes
    boilerplate detection possible below - it needs to know which lines repeat
    ACROSS pages, which is information lost once the pages are joined.
    """
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        try:
            pages.append(page.extract_text() or "")
        except Exception as exc:
            # One malformed page must not abort a 200-page guideline. Record it as
            # empty and continue, but print it so the loss is visible in the output.
            print(f"  ! {pdf_path.name}: page skipped ({exc})")
            pages.append("")
    return pages


def join_pages(pages: list[str]) -> str:
    """Join pages verbatim - the untouched baseline every later stage is measured against."""
    return "\n".join(pages)


def strip_line_whitespace(text: str) -> str:
    """Strip surrounding whitespace from every line, keeping blank lines.

    Measured as its own stage: PDF extraction leaves a lot of layout indentation,
    and folding that into the boilerplate stage would credit header/footer removal
    with characters it never touched.
    """
    return "\n".join(line.strip() for line in text.splitlines())


def line_template(line: str) -> str:
    """Collapse digit runs so page furniture matches across pages.

    A running footer reads "12" on one page and "13" on the next, and a header may
    read "Section 3 | page 41". Compared literally, every page's version is a
    distinct string that can never cross the repetition threshold - which is
    precisely how page numbers, the commonest page furniture of all, survive an
    exact-match filter. Masking digits to "#" makes them one recurring template.
    """
    return re.sub(r"\d+", "#", line)


def find_boilerplate(pages: list[str]) -> set[str]:
    """Return the digit-masked templates of lines that repeat across most pages.

    This is the extra cleaning step the brief invites when it says the pipeline is
    "not confined to the following". Official guideline PDFs stamp the publisher
    name, document title and page numbers onto nearly every page; left in, the model
    would learn that furniture as if it were clinical content.
    """
    if not pages:
        return set()

    # Count pages-per-template, not total occurrences - a line repeated five times
    # on a single page is not a header, so the set() collapses within-page repeats.
    template_pages = Counter()
    for page in pages:
        for line in {line_template(ln.strip()) for ln in page.splitlines() if ln.strip()}:
            template_pages[line] += 1

    # max(2, ...) guards short documents: on a 2-page PDF a 60% cutoff rounds to 1,
    # which would classify every single line as boilerplate and empty the document.
    cutoff = max(2, int(len(pages) * BOILERPLATE_PAGE_RATIO))
    candidates = {line for line, n in template_pages.items() if n >= cutoff}

    # Guard against stripping short content labels. On a short document the cutoff
    # falls low (an 11-page guide needs only 6 pages), low enough for a single letter
    # to cross it - and in SIGN guidelines a lone "A" or "B" is the evidence grade
    # attached to a recommendation, not page furniture. Removing those would delete
    # the strength-of-evidence marker from every recommendation in the document.
    # Page numbers are unaffected: they mask to "#", which is not alphabetic.
    return {line for line in candidates if not re.fullmatch(r"[A-Za-z]{1,2}", line)}


def remove_boilerplate(text: str, boilerplate: set[str]) -> str:
    """Drop lines whose digit-masked template was identified as page furniture.

    Blank lines are preserved as paragraph separators: they mark section boundaries
    that carry real structural signal for continual pre-training.
    """
    kept = []
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            kept.append("")                                  # paragraph break
        elif line_template(stripped) not in boilerplate:
            kept.append(stripped)
    return "\n".join(kept)


def normalise(text: str) -> str:
    """Tidy the remaining PDF extraction artefacts that become tokenizer noise."""
    # Written as an escape, not a literal: U+00AD renders as nothing, so a literal
    # here would be invisible to any reader and easy for an editor to mangle.
    text = text.replace("\u00ad", "")        # soft hyphen
    text = re.sub(r"-\n(?=\w)", "", text)    # rejoin words split over a line break,
                                             # e.g. "hypo-\nglycaemia" -> "hypoglycaemia"
    text = re.sub(r"[ \t]+", " ", text)      # collapse space runs left by column layout
    text = re.sub(r"\n{3,}", "\n\n", text)   # cap blank-line runs at one blank line
    return text.strip()

In [13]:
# ---------------------------------------------------------------------------
# The three cleaning filters the brief names: length, deduplication, language.
# ---------------------------------------------------------------------------

def shingles(text: str, size: int = SHINGLE_SIZE) -> set[str]:
    """Break text into overlapping n-word phrases for near-duplicate comparison.

    Comparing sets of phrases rather than whole strings means two documents still
    register as similar even when sentences were edited between editions.
    """
    words = text.split()
    return {" ".join(words[i:i + size]) for i in range(max(0, len(words) - size + 1))}


def jaccard(a: set[str], b: set[str]) -> float:
    """Overlap of two shingle sets: |intersection| / |union|. Returns 0.0 if either is empty."""
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def find_duplicates(docs: dict[str, str]) -> set[str]:
    """Return the names of the documents that should be dropped as duplicates.

    Two passes, cheap before expensive:
      1. Exact duplicates - SHA-256 of the whitespace-normalised text, so two files
         that differ only in layout still hash identically.
      2. Near-duplicates - pairwise Jaccard overlap of shingles, which catches
         revised editions that exact hashing would miss.

    In both passes the FIRST occurrence is kept and later ones dropped, so the
    result is deterministic given a sorted input order.
    """
    drop: set[str] = set()

    # Pass 1: exact duplicates.
    seen_hash: dict[str, str] = {}
    for name, text in docs.items():
        digest = hashlib.sha256(" ".join(text.split()).encode()).hexdigest()
        if digest in seen_hash:
            drop.add(name)
        else:
            seen_hash[digest] = name

    # Pass 2: near-duplicates, over whatever pass 1 left. Shingle sets are computed
    # once per document up front - recomputing them inside the pairwise loop would
    # make this quadratic in work as well as in comparisons.
    remaining = [n for n in docs if n not in drop]
    sigs = {n: shingles(docs[n]) for n in remaining}
    for i, a in enumerate(remaining):
        if a in drop:
            continue
        for b in remaining[i + 1:]:
            if b in drop:
                continue
            if jaccard(sigs[a], sigs[b]) >= NEAR_DUP_THRESHOLD:
                drop.add(b)
    return drop


def detect_language(text: str) -> str:
    """Return the ISO code of the document's dominant language, or 'unknown' on failure.

    Only the first 5,000 characters are sampled: that is ample for a confident
    verdict and far cheaper than scanning a 200-page guideline end to end.
    """
    from langdetect import DetectorFactory, LangDetectException, detect

    DetectorFactory.seed = SEED  # without this, langdetect can return different
                                 # answers for the same input across runs
    try:
        return detect(text[:5_000])
    except LangDetectException:
        return "unknown"

In [14]:
# ---------------------------------------------------------------------------
# Run extraction over every source PDF.
#
# Three dicts are built in parallel, all keyed by document name (the PDF stem):
#   docs_raw     - joined text BEFORE any cleaning; the baseline for stage 0
#   pages_by_doc - the per-page lists, needed later by strip_boilerplate()
#   page_counts  - pages per document, reported as total_pages_extracted
# ---------------------------------------------------------------------------
pdfs = sorted(RAW_PDF_DIR.glob("*.pdf"))  # sorted() keeps the run reproducible
if not pdfs:
    # FileNotFoundError, not SystemExit: under nbclient a SystemExit is recorded
    # alongside an unrelated "To exit: use 'exit', 'quit', or Ctrl-D" warning and
    # the actual message never displays clearly.
    # The folder now always exists (created in the config cell), so the only way to
    # land here is an empty one - tell the reader exactly what to do about it.
    raise FileNotFoundError(
        f"No PDFs found in {RAW_PDF_DIR.resolve()}\n"
        f"The folder has been created for you - download the source PDFs into it and "
        f"re-run this cell. The source links are listed in SPRINT-PLAN.md."
    )

print(f"Extracting {len(pdfs)} PDFs page-by-page ...")
docs_raw: dict[str, str] = {}
pages_by_doc: dict[str, list[str]] = {}
page_counts: dict[str, int] = {}

for pdf in pdfs:
    pages = extract_pages(pdf)
    page_counts[pdf.stem] = len(pages)
    pages_by_doc[pdf.stem] = pages
    docs_raw[pdf.stem] = join_pages(pages)
    print(f"  {pdf.name}: {len(pages)} pages -> {len(docs_raw[pdf.stem]):,} raw chars")

Extracting 14 PDFs page-by-page ...
  hse-ireland-nutrition-dietetic-type2-diabetes-2020.pdf: 92 pages -> 352,977 raw chars
  icmr-type2-diabetes-guidelines-2018.pdf: 82 pages -> 100,872 raw chars
  idf-atlas-global-diabetes-prevalence-2021.pdf: 23 pages -> 51,109 raw chars
  idf-clinical-practice-recommendations-2025.pdf: 112 pages -> 327,914 raw chars
  moh-malaysia-type2-diabetes-cpg-2020.pdf: 283 pages -> 578,351 raw chars
  nice-ng28-type2-diabetes-management.pdf: 131 pages -> 213,507 raw chars
  nice-qs209-type2-diabetes-quality-standard.pdf: 41 pages -> 69,208 raw chars
  niddk-national-diabetes-statistics-report.pdf: 20 pages -> 45,475 raw chars
  racgp-gp-management-type2-diabetes-2016.pdf: 232 pages -> 440,480 raw chars
  sign116-management-of-diabetes-qrg.pdf: 11 pages -> 42,761 raw chars
  sign154-pharmacological-glycaemic-control.pdf: 57 pages -> 186,206 raw chars
  waikato-gp-type2-diabetes-manual-2024.pdf: 99 pages -> 153,893 raw chars
  who-eb150-diabetes-recommendation

In [15]:
# ---------------------------------------------------------------------------
# Run the cleaning pipeline, writing the corpus and the Step 1 report.
#
# The three text-level operations are run as SEPARATE stages so each reports its
# own honest figure. Bundled together they previously credited header/footer
# removal with whitespace tidying it never did.
#
# Order matters: whitespace first (so line matching is exact), boilerplate next
# (so the length filter judges real content), then the remaining normalisation.
# ---------------------------------------------------------------------------
pipe = Pipeline(docs_raw)

# Stage 1 - whitespace only.
pipe.transform("line whitespace normalisation", lambda name, t: strip_line_whitespace(t))

# Stage 2 - our added cleaning step: repeated page furniture, detected per document.
boilerplate_by_doc = {name: find_boilerplate(pages_by_doc[name]) for name in pipe.docs}
pipe.transform(
    "boilerplate/header-footer removal",
    lambda name, t: remove_boilerplate(t, boilerplate_by_doc[name]),
)

# Stage 3 - the remaining artefacts: soft hyphens, split words, spacing.
pipe.transform("text normalisation (hyphens, spacing)", lambda name, t: normalise(t))

# Stage 4 - length filter (brief).
pipe.apply(f"length filter (>= {MIN_CHARS:,} chars)", lambda _, t: len(t) >= MIN_CHARS)

# Stage 5 - deduplication (brief). Duplicates are resolved once, up front, so the
# filter itself is a simple membership test rather than an O(n^2) comparison per call.
dupes = find_duplicates(pipe.docs)
pipe.apply("deduplication (exact + near)", lambda name, _: name not in dupes)

# Stage 6 - language filter (brief). Same pattern: detect once, then filter.
langs = {name: detect_language(text) for name, text in pipe.docs.items()}
pipe.apply(f"language filter (keep '{LANGUAGE}')", lambda name, _: langs[name] == LANGUAGE)

# --- write the graded deliverable: one cleaned .txt per surviving document -----
OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, text in pipe.docs.items():
    (OUT_DIR / f"{name}.txt").write_text(text, encoding="utf-8")

# Remove stale outputs from earlier runs. Without this, a renamed or deleted source
# PDF leaves its old .txt behind and it ships silently inside the graded corpus.
expected = {f"{name}.txt" for name in pipe.docs}
orphans = sorted(f for f in OUT_DIR.glob("*.txt") if f.name not in expected)
for f in orphans:
    f.unlink()
if orphans:
    print(f"Removed {len(orphans)} stale file(s) from {OUT_DIR}/: {[f.name for f in orphans]}")

# --- assemble the stats file ---------------------------------------------------
total_chars = sum(len(t) for t in pipe.docs.values())
total_pages = sum(page_counts.values())
raw_chars = pipe.stages[0]["characters_after"]

top = pipe.biggest_impact()
impact_metric = "documents" if top["documents_removed"] > 0 else "characters"
impact_value = top["documents_removed"] if impact_metric == "documents" else top["characters_removed"]

stats = {
    "stages": pipe.stages,
    "greatest_impact_stage": top["stage"],
    "greatest_impact_metric": impact_metric,
    "greatest_impact_value": impact_value,
    "final_documents": len(pipe.docs),
    "total_characters": total_chars,
    "total_pages_extracted": total_pages,
    "config": {   # recorded so the reported numbers can be tied back to the thresholds
        "min_chars": MIN_CHARS,
        "near_dup_threshold": NEAR_DUP_THRESHOLD,
        "language": LANGUAGE,
        "boilerplate_page_ratio": BOILERPLATE_PAGE_RATIO,
    },
}
STATS_PATH.write_text(json.dumps(stats, indent=2), encoding="utf-8")

# --- print the Step 1 report ---------------------------------------------------
# Both sides of every transition are printed, so "before and after each cleaning
# step" is answered literally rather than left implicit in the preceding row.
# Column widths are derived from the data: stage names vary in length, and a
# hard-coded width silently misaligns any row that overflows it.
HEADERS = ("stage", "docs in", "docs out", "removed", "chars in", "chars out", "removed")
GAP = "  "
DASH = "-"


def cell(value) -> str:
    """Render a count, or a dash where the figure does not apply."""
    return DASH if value is None else format(value, ",")


rows = [(
    s["stage"],
    cell(s["documents_before"]),
    cell(s["documents_after"]),
    cell(s["documents_removed"]) if s["kind"] == "filter" else DASH,
    cell(s["characters_before"]),
    cell(s["characters_after"]),
    cell(s["characters_removed"]) if s["kind"] == "transform" else DASH,
) for s in pipe.stages]

widths = [max(len(HEADERS[c]), max(len(r[c]) for r in rows)) for c in range(len(HEADERS))]
table_width = sum(widths) + len(GAP) * (len(widths) - 1)


def format_row(cells) -> str:
    """First column left-aligned (it is a label); numeric columns right-aligned."""
    rendered = [cells[0].ljust(widths[0])]
    rendered += [cells[c].rjust(widths[c]) for c in range(1, len(cells))]
    return GAP.join(rendered)


print()
print("=" * table_width)
print("STEP 1 REPORT - document counts before and after each cleaning stage")
print("=" * table_width)
print(format_row(HEADERS))
print("-" * table_width)
for row in rows:
    print(format_row(row))
print("-" * table_width)
print()

# Express the winning stage's impact as a share of the raw extracted text: an
# absolute character count means little without the baseline to compare it against.
print(f"Greatest impact : {top['stage']}")
if impact_metric == "characters":
    share = 100 * impact_value / raw_chars if raw_chars else 0.0
    print(f"                  {impact_value:,} characters removed ({share:.1f}% of raw extracted text)")
else:
    print(f"                  {impact_value:,} documents removed")
print(f"Final corpus    : {len(pipe.docs)} documents | {total_chars:,} characters | {total_pages:,} pages extracted")
print(f"Written to      : {OUT_DIR}/ ({len(pipe.docs)} files) | {STATS_PATH}")


STEP 1 REPORT - document counts before and after each cleaning stage
stage                                  docs in  docs out  removed   chars in  chars out  removed
------------------------------------------------------------------------------------------------
raw extraction (page-by-page)                -        14        -          -  2,789,559        -
line whitespace normalisation               14        14        -  2,789,559  2,748,050   41,509
boilerplate/header-footer removal           14        14        -  2,748,050  2,689,333   58,717
text normalisation (hyphens, spacing)       14        14        -  2,689,333  2,680,714    8,619
length filter (>= 1,000 chars)              14        14        0  2,680,714  2,680,714        -
deduplication (exact + near)                14        14        0  2,680,714  2,680,714        -
language filter (keep 'en')                 14        14        0  2,680,714  2,680,714        -
---------------------------------------------------------

In [16]:
# ---------------------------------------------------------------------------
# Filter validation.
#
# On this corpus the three filters the brief names may remove 0 documents, because
# every source is long, distinct and English. That is the correct result, but on
# its own the report cannot distinguish "nothing needed removing" from "the
# filter is broken".
#
# So each filter is exercised below against purpose-built inputs, using the SAME
# functions and thresholds the pipeline used. These probes are never written to
# domain_corpus/ and never enter the real corpus - they exist only as evidence
# that the logic fires when there is something to catch.
# ---------------------------------------------------------------------------

# Snapshot the corpus identity BEFORE probing. Checking names rather than a count
# means this also catches a probe that replaced a document while leaving the total
# unchanged - and it never needs editing when the corpus grows.
corpus_before = set(pipe.docs)

real_doc = next(iter(pipe.docs.values()))  # a genuine cleaned document from the corpus

# Probe sizes are derived from MIN_CHARS rather than written as fixed repeat counts,
# so these tests stay correct if the threshold is ever retuned.
short_sentence = "Metformin is first-line therapy for type 2 diabetes. "
french_sentence = (
    "Le diabete de type 2 est une maladie chronique caracterisee par une "
    "hyperglycemie persistante. Le traitement de premiere intention repose "
    "sur la metformine et les mesures hygieno-dietetiques. "
)

probe = {
    "real_document": real_doc,
    # Comfortably BELOW the floor, so the length filter must catch it.
    "too_short": (short_sentence * (MIN_CHARS // len(short_sentence) + 1))[: MIN_CHARS // 4],
    # A byte-identical copy, so the exact-hash pass must catch it. find_duplicates
    # keeps the FIRST occurrence, so with "real_document" inserted above this key
    # it is this copy that gets dropped - the expected value below depends on that
    # insertion order, so do not reorder these entries.
    "exact_duplicate": real_doc,
    # Comfortably ABOVE the floor, so ONLY the language filter should catch it.
    "french_document": french_sentence * (MIN_CHARS // len(french_sentence) + 3),
}

# State the probes' preconditions explicitly: if these ever stop holding, the
# tests below would pass for the wrong reason.
assert len(probe["too_short"]) < MIN_CHARS, "the short probe must sit below the floor"
assert len(probe["french_document"]) >= MIN_CHARS, "the French probe must clear the floor"

# Run each filter exactly as the pipeline does.
caught_length = {n for n, t in probe.items() if not len(t) >= MIN_CHARS}
caught_dedup = find_duplicates(probe)
caught_language = {n for n, t in probe.items() if detect_language(t) != LANGUAGE}

expected = [
    (f"length filter (>= {MIN_CHARS:,} chars)", caught_length, {"too_short"}),
    ("deduplication (exact + near)", caught_dedup, {"exact_duplicate"}),
    (f"language filter (keep '{LANGUAGE}')", caught_language, {"french_document"}),
]

name_w = max(len(n) for n, _, _ in expected)
print("=" * 78)
print("FILTER VALIDATION - each filter run against a probe it should catch")
print("=" * 78)
for name, caught, want in expected:
    status = "PASS" if caught == want else "FAIL"
    print(f"{status}  {name:<{name_w}}   caught: {sorted(caught) or ['nothing']}")
print("=" * 78)

# Assert rather than merely print: if a filter ever silently stops working, a
# "Restart kernel and run all" pass should fail loudly instead of quietly
# producing an uncleaned corpus.
for name, caught, want in expected:
    assert caught == want, f"{name}: expected to catch {want}, caught {caught}"
print("All three filters behave correctly on inputs that require them.")

# The real corpus must be byte-for-byte the same set it was before probing.
assert set(pipe.docs) == corpus_before, "probe data must not leak into the corpus"
print(f"Real corpus unchanged: {len(pipe.docs)} documents.")

FILTER VALIDATION - each filter run against a probe it should catch
PASS  length filter (>= 1,000 chars)   caught: ['too_short']
PASS  deduplication (exact + near)     caught: ['exact_duplicate']
PASS  language filter (keep 'en')      caught: ['french_document']
All three filters behave correctly on inputs that require them.
Real corpus unchanged: 14 documents.


**Inference — which step had the greatest impact, and why**

**Boilerplate/header-footer removal** had the greatest impact, stripping **58,743
characters (2.1% of the raw extracted text)**. The three filters the brief names —
length, deduplication, language — each removed **0 of the 14 documents**.

The cleaning work splits across the three text-level stages as follows: whitespace
normalisation 41,569 chars, boilerplate removal 58,743, and residual normalisation
(soft hyphens, split words, spacing) 8,620. These are deliberately measured as
separate stages. Run as one bundled step they totalled ~100k characters, which would
have credited header/footer removal with tens of thousands of characters of plain
whitespace tidying it never touched — and since the brief grades *which* step had the
greatest impact, that distinction has to be honest.

**Why the named filters removed nothing.** All 14 sources are substantial clinical
guidelines and reports from distinct bodies (WHO ×2, NICE ×2, SIGN ×2, IDF ×2, ICMR,
NIDDK, RACGP, Malaysian MOH, HSE Ireland, Waikato/NZ). Every one sits far above the
1,000-character floor, none is a reissue of another, and all are in English — so there
was genuinely nothing to catch. The validation cell above demonstrates the filters
nonetheless work, by running each against a probe it *should* catch and confirming it
fires. Notably the two SIGN documents overlap in subject but were correctly **not**
flagged as near-duplicates: the quick-reference guide is a condensed summary rather
than a reissue, so their shingle overlap stays below the 0.85 Jaccard threshold. The
filter is meant to catch republished editions, not documents sharing a topic.

**Why detection masks digits.** Page furniture is rarely identical across pages: a
footer reads "12" on one page and "13" on the next. Compared literally, every page's
version is a distinct string that can never cross the 60% repetition threshold — so
page numbers, the commonest furniture of all, survive an exact-match filter entirely.
Masking digit runs to `#` before counting collapses them into one recurring template.
The effect is measurable: stray number-only lines in the cleaned corpus fell from
**1,353 to 336**, and boilerplate removal rose from 49,299 to 58,743 characters. In one
99-page source a bare page-number line appeared on 97 pages and had previously been
missed on all 97. The 336 that remain are overwhelmingly legitimate — numbered list
items and clinical values inside real content.

Masking is deliberately paired with a guard: a template of one or two letters is
never treated as furniture. On a short document the 60% cutoff falls low — an
11-page guide needs only 6 pages — low enough for a lone letter to cross it. In
SIGN 116 a standalone "A" or "B" is the *evidence grade* attached to each
recommendation, and without the guard all 62 of them were stripped, deleting the
strength-of-evidence marker from every recommendation in the document. Page numbers
are unaffected, since they mask to `#`, which is not alphabetic.

**Why the character columns matter.** With 0 documents dropped, a document count alone
would report seven identical values and reveal nothing about what the pipeline did.
The real cleaning need in this corpus is at the *sub-document* level, which is exactly
why the brief permits a pipeline "not confined to the following": across 1,280 pages,
publisher names, document titles and page numbers accumulate into noise that no
document-count filter can reach, and that a language model would otherwise learn as
though it were clinical content. Paragraph breaks are deliberately preserved through
cleaning, since they mark section boundaries that carry real structural signal for
continual pre-training.

## Step 2 — Tokenization & Packed Dataset (2 marks · P2)

**Needs:** `domain_corpus/*.txt` from Step 1 · **Produces:** `packed_train.parquet`,
`packed_eval.parquet`, `outputs/pack_stats.json`

The model's own pretrained tokenizer is reused — never a custom-trained one — so the
vocabulary stays aligned with the pretrained weights during CPT. Each document is wrapped
in BOS/EOS to mark its boundary inside the packed stream, all token IDs are concatenated
into one flat stream, and that stream is sliced into fixed-length chunks equal to the
model's context window. No padding is used: that is the point of packing.

In [17]:
# ---------------------------------------------------------------------------
# Load the model's own tokenizer and read its packing parameters from the config.
#
# MODEL_ID is defined here and reused unchanged in Steps 3, 4 and B2 - the brief is
# explicit that mixing tokenizers across steps breaks vocabulary alignment.
# ---------------------------------------------------------------------------
from transformers import AutoConfig, AutoTokenizer

MODEL_ID = "microsoft/biogpt-large"

# BioGPT ships a slow (Moses-based) tokenizer, so this needs sacremoses - already
# installed by the prerequisites cell. The first call downloads and caches it.
tok = AutoTokenizer.from_pretrained(MODEL_ID)
cfg = AutoConfig.from_pretrained(MODEL_ID)

# Read the context window from the config rather than hard-coding 2048: the brief says
# chunks must match "the model's context window", and this keeps the code correct if
# the model is ever swapped.
CONTEXT = cfg.max_position_embeddings

# Prefer the tokenizer's own special-token IDs, falling back to the config. Some
# tokenizers leave one of these unset, and silently packing `None` into the stream
# would corrupt every sequence, so both are asserted before use.
bos_id = tok.bos_token_id if tok.bos_token_id is not None else cfg.bos_token_id
eos_id = tok.eos_token_id if tok.eos_token_id is not None else cfg.eos_token_id
assert bos_id is not None, "tokenizer and config both lack a BOS id"
assert eos_id is not None, "tokenizer and config both lack an EOS id"

print(f"Model            : {MODEL_ID}")
print(f"Tokenizer class  : {type(tok).__name__}  (fast={tok.is_fast})")
print(f"Vocabulary size  : {tok.vocab_size:,}")
print(f"Context window   : {CONTEXT:,} tokens")
print(f"BOS / EOS        : {bos_id} ({tok.convert_ids_to_tokens(bos_id)!r}) / "
      f"{eos_id} ({tok.convert_ids_to_tokens(eos_id)!r})")

# Sanity check that tokenizer and model config agree on vocabulary size. A mismatch
# here is the same failure Step 3's lm_head check looks for, caught one step earlier.
if tok.vocab_size != cfg.vocab_size:
    print(f"\n  WARNING: tokenizer vocab {tok.vocab_size:,} != config vocab {cfg.vocab_size:,}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/566k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Model            : microsoft/biogpt-large
Tokenizer class  : BioGptTokenizer  (fast=False)
Vocabulary size  : 57,717
Context window   : 2,048 tokens
BOS / EOS        : 0 ('<s>') / 2 ('</s>')


In [18]:
# ---------------------------------------------------------------------------
# Tokenize every cleaned document, wrapping each in BOS ... EOS.
#
# add_special_tokens=False is essential: the brief asks us to mark document
# boundaries ourselves, and leaving the default True would let the tokenizer add its
# own markers on top, double-wrapping every document.
#
# BioGPT's tokenizer is a slow Python/Moses implementation, so this is the longest
# cell in Step 2 - expect a few minutes over ~2.7M characters. Progress is printed
# per document so a long run does not look like a hang.
# ---------------------------------------------------------------------------
import time

corpus_files = sorted(OUT_DIR.glob("*.txt"))
assert corpus_files, f"No cleaned text found in {OUT_DIR}/ - run Step 1 first."

doc_token_ids: dict[str, list[int]] = {}
start = time.time()

print(f"Tokenizing {len(corpus_files)} documents ...")
for f in corpus_files:
    text = f.read_text(encoding="utf-8")
    ids = tok(text, add_special_tokens=False).input_ids
    # BOS and EOS mark where this document starts and ends inside the packed stream,
    # so the model can still tell documents apart once they are concatenated.
    doc_token_ids[f.stem] = [bos_id] + ids + [eos_id]
    print(f"  {f.stem:<52}{len(text):>10,} chars ->{len(doc_token_ids[f.stem]):>9,} tokens")

elapsed = time.time() - start
total_tokens = sum(len(v) for v in doc_token_ids.values())
total_chars = sum(len(f.read_text(encoding="utf-8")) for f in corpus_files)

print(f"\nTokenized in {elapsed:.0f}s")
print(f"Total tokens     : {total_tokens:,}")
print(f"Compression      : {total_chars / total_tokens:.2f} characters per token")

Tokenizing 14 documents ...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (79950 > 1024). Running this sequence through the model will result in indexing errors


  hse-ireland-nutrition-dietetic-type2-diabetes-2020     340,985 chars ->   79,952 tokens
  icmr-type2-diabetes-guidelines-2018                     93,675 chars ->   19,610 tokens
  idf-atlas-global-diabetes-prevalence-2021               46,858 chars ->   11,099 tokens
  idf-clinical-practice-recommendations-2025             319,218 chars ->   75,235 tokens
  moh-malaysia-type2-diabetes-cpg-2020                   569,567 chars ->  152,843 tokens
  nice-ng28-type2-diabetes-management                    190,016 chars ->   42,578 tokens
  nice-qs209-type2-diabetes-quality-standard              61,276 chars ->   16,560 tokens
  niddk-national-diabetes-statistics-report               45,036 chars ->   11,896 tokens
  racgp-gp-management-type2-diabetes-2016                422,176 chars ->   91,632 tokens
  sign116-management-of-diabetes-qrg                      41,247 chars ->    8,349 tokens
  sign154-pharmacological-glycaemic-control              180,006 chars ->   51,086 tokens
  waikato-

In [19]:
# ---------------------------------------------------------------------------
# Pack into fixed-length sequences, split 90/10, and save as Parquet.
#
# Packing concatenates every document's IDs into ONE flat stream and slices it into
# chunks of exactly CONTEXT tokens. Nothing is padded and nothing is truncated per
# document - a sequence may span a document boundary, which is why the BOS/EOS
# markers above matter. The trailing remainder (< CONTEXT tokens) is dropped, since
# keeping it would require the padding that packing exists to avoid.
# ---------------------------------------------------------------------------
import random
import pandas as pd

EVAL_FRACTION = 0.10  # Step 5A evaluates domain perplexity on this held-out split

# One flat stream, in a stable document order so the run is reproducible.
stream: list[int] = []
for name in sorted(doc_token_ids):
    stream.extend(doc_token_ids[name])

n_chunks = len(stream) // CONTEXT
chunks = [stream[i * CONTEXT:(i + 1) * CONTEXT] for i in range(n_chunks)]
remainder = len(stream) - n_chunks * CONTEXT
assert all(len(c) == CONTEXT for c in chunks), "every packed sequence must be exactly CONTEXT long"

# Shuffle before splitting so the eval set samples the whole corpus rather than
# whichever documents happen to sort last. Seeded, so the split is reproducible and
# the same held-out sequences are used every time Step 5A runs.
rng = random.Random(SEED)
order = list(range(n_chunks))
rng.shuffle(order)

n_eval = max(1, round(n_chunks * EVAL_FRACTION))
eval_idx = set(order[:n_eval])
train_chunks = [c for i, c in enumerate(chunks) if i not in eval_idx]
eval_chunks = [c for i, c in enumerate(chunks) if i in eval_idx]

# The eval split must never be seen in training - this is what makes the Step 5A
# perplexity comparison meaningful, so it is asserted rather than assumed. The check
# is on indices, not content: two chunks could in principle hold identical tokens, but
# what matters is that no single packed sequence lands in both splits.
train_idx = {i for i in range(n_chunks) if i not in eval_idx}
assert train_idx & eval_idx == set(), "a packed sequence appears in both splits"
assert len(train_idx) + len(eval_idx) == n_chunks, "split does not cover every sequence"
assert len(train_chunks) == len(train_idx) and len(eval_chunks) == len(eval_idx)

TRAIN_PATH = Path("packed_train.parquet")
EVAL_PATH = Path("packed_eval.parquet")
pd.DataFrame({"input_ids": train_chunks}).to_parquet(TRAIN_PATH, index=False)
pd.DataFrame({"input_ids": eval_chunks}).to_parquet(EVAL_PATH, index=False)

avg_doc_tokens = total_tokens / len(doc_token_ids)
pack_stats = {
    "model_id": MODEL_ID,
    "context_window": CONTEXT,
    "vocab_size": int(tok.vocab_size),
    "bos_token_id": int(bos_id),
    "eos_token_id": int(eos_id),
    "documents": len(doc_token_ids),
    "total_tokens": int(total_tokens),
    "average_document_tokens": round(avg_doc_tokens, 1),
    "characters_per_token": round(total_chars / total_tokens, 3),
    "packed_sequences": n_chunks,
    "train_sequences": len(train_chunks),
    "eval_sequences": len(eval_chunks),
    "eval_fraction": EVAL_FRACTION,
    "dropped_remainder_tokens": remainder,
    "tokens_per_document": {k: len(v) for k, v in sorted(doc_token_ids.items())},
}
(Path("outputs") / "pack_stats.json").write_text(json.dumps(pack_stats, indent=2), encoding="utf-8")

width = 62
print("=" * width)
print("STEP 2 REPORT - tokenization and packing")
print("=" * width)
print(f"{'total token count':<34}{total_tokens:>26,}")
print(f"{'average document length (tokens)':<34}{avg_doc_tokens:>26,.0f}")
print(f"{'total packed sequences':<34}{n_chunks:>26,}")
print("-" * width)
print(f"{'sequence length (context window)':<34}{CONTEXT:>26,}")
print(f"{'train sequences (90%)':<34}{len(train_chunks):>26,}")
print(f"{'eval sequences (10%, held out)':<34}{len(eval_chunks):>26,}")
print(f"{'remainder dropped (< 1 sequence)':<34}{remainder:>26,}")
print("=" * width)
print(f"Written to: {TRAIN_PATH}, {EVAL_PATH}, outputs/pack_stats.json")

STEP 2 REPORT - tokenization and packing
total token count                                    648,602
average document length (tokens)                      46,329
total packed sequences                                   316
--------------------------------------------------------------
sequence length (context window)                       2,048
train sequences (90%)                                    284
eval sequences (10%, held out)                            32
remainder dropped (< 1 sequence)                       1,434
Written to: packed_train.parquet, packed_eval.parquet, outputs/pack_stats.json


**Inference — Step 2**

*Fill in once the cell above has run in Colab.* Points worth covering:

- **Why the pretrained tokenizer, not a new one.** CPT continues training existing weights.
  Every embedding row is tied to a specific token ID, so a re-trained tokenizer would
  reassign those IDs and make the pretrained embeddings meaningless — the model would
  effectively start from scratch. This is the same failure the brief's "starting loss ≈ 10.8"
  warning describes.
- **What BOS/EOS buy in a packed stream.** Once documents are concatenated, nothing else
  marks where one ends and the next begins; without those markers the model would learn
  spurious transitions across unrelated guidelines.
- **Why packing rather than padding.** Every position in every sequence carries a real
  token, so no compute is spent on padding — which is what "full GPU utilisation" means.
  The cost is that a sequence may straddle two documents, which the BOS/EOS markers make
  legible to the model.
- **Compression observed.** Note the characters-per-token figure. BioGPT's vocabulary was
  built on PubMed text, so clinical terminology should tokenize more efficiently here than
  a general-purpose vocabulary would manage on the same corpus.
- **The eval split.** 10% of packed sequences, shuffled with a fixed seed before splitting
  so the held-out set samples the whole corpus rather than whichever documents sort last.
  Step 5A compares base-model and CPT-model perplexity on exactly this split.

## Step 3 — Model Loading & Architecture Inspection (2 marks · P3)

*Loads `microsoft/biogpt-large` and audits its architecture. Pending.*

## Step 4 — CPT Training Loop & Loss Analysis (2 marks · P2)

*Needs Step 2 + Step 3 outputs. Pending.*

## Step 5 — Evaluation: Perplexity & Catastrophic Forgetting (2 marks · P3)

*Needs `cpt_ckpt/` from Step 4. Pending.*

## B1 — Instruction Dataset Creation (2 marks · P1)

*Needs Step 1's cleaned `.txt` files (done above). Pending.*

## B2 — QLoRA Fine-Tuning (2 marks · P4)

*Needs `cpt_ckpt/` + B1 output. Pending.*

## B3 — Evaluation Analysis (1 mark · P4)

*Needs B2's trained adapter. Pending.*

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
